In [1]:
import pandas as pd

url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"

df = pd.read_csv(url, compression="zip", sep=";", na_values="?")

print(df.shape)
df.head()


(2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [2]:
df = df.dropna().copy()

df['Datetime'] = pd.to_datetime( df['Date'] + ' ' + df['Time'],dayfirst=True)

df = df.drop(columns=['Date', 'Time'])

print("Cleaned Shape:", df.shape)
df.head()


Cleaned Shape: (2049280, 8)


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [3]:

df = df.set_index('Datetime')
hourly_df = df.resample('h').mean().dropna()
hourly_df = hourly_df.reset_index()

print("Hourly Dataset Shape:", hourly_df.shape)
hourly_df.head()


Hourly Dataset Shape: (34168, 8)


,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,0.527778,16.861111
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,6.716667,16.866667
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,1.433333,16.683333
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.000000,16.783333
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,0.416667,17.216667


In [4]:

from sklearn.ensemble import IsolationForest

X = hourly_df.drop(columns=['Datetime'])

model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X)

hourly_df['Anomaly'] = model.predict(X)

hourly_df['Anomaly'] = hourly_df['Anomaly'].map({
    1: 'Normal',
    -1: 'Anomaly'
})

print(hourly_df['Anomaly'].value_counts())


Anomaly
Normal     32459
Anomaly     1709
Name: count, dtype: int64


In [5]:

# Calculate anomaly scores
hourly_df['Anomaly_Score'] = -model.decision_function(X)

# Assign risk levels
hourly_df['Risk_Level'] = 'Low'

anomalies = hourly_df['Anomaly'] == 'Anomaly'

cutoff = hourly_df.loc[anomalies, 'Anomaly_Score'].median()

hourly_df.loc[anomalies, 'Risk_Level'] = 'Medium'

hourly_df.loc[
    anomalies & (hourly_df['Anomaly_Score'] >= cutoff),
    'Risk_Level'
] = 'High'

print(hourly_df['Risk_Level'].value_counts())


Risk_Level
Low       32459
High        855
Medium      854
Name: count, dtype: int64


In [6]:

def recommend(row):
    suggestions = []

    if row['Global_active_power'] > 3:
        suggestions.append("Review high electricity consumption")

    if row['Global_intensity'] > 15:
        suggestions.append("Check high-current appliances")

    if row['Datetime'].hour < 6 and row['Global_active_power'] > 2:
        suggestions.append("Review overnight electricity usage")

    if row['Risk_Level'] == 'High':
        suggestions.append("Prioritize investigation of unusual readings")

    if not suggestions:
        suggestions.append("Continue routine energy monitoring")

    return "; ".join(suggestions)

hourly_df['Recommendation'] = hourly_df.apply(recommend, axis=1)

hourly_df[['Datetime', 'Global_active_power',
           'Anomaly', 'Risk_Level', 'Recommendation']].head(10)


,Datetime,Global_active_power,Anomaly,Risk_Level,Recommendation
0,2006-12-16 17:00:00,4.222889,Anomaly,High,Review high electricity consumption; Check hig...
1,2006-12-16 18:00:00,3.632200,Anomaly,Medium,Review high electricity consumption; Check hig...
2,2006-12-16 19:00:00,3.400233,Normal,Low,Review high electricity consumption
3,2006-12-16 20:00:00,3.268567,Normal,Low,Review high electricity consumption
4,2006-12-16 21:00:00,3.056467,Normal,Low,Review high electricity consumption
5,2006-12-16 22:00:00,2.200133,Normal,Low,Continue routine energy monitoring
6,2006-12-16 23:00:00,2.061600,Normal,Low,Continue routine energy monitoring
7,2006-12-17 00:00:00,1.882467,Normal,Low,Continue routine energy monitoring
8,2006-12-17 01:00:00,3.349400,Anomaly,High,Review high electricity consumption; Review ov...
9,2006-12-17 02:00:00,1.587267,Normal,Low,Continue routine energy monitoring


In [9]:

from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Scale features for distance-based evaluation
X_scaled = StandardScaler().fit_transform(X)

# Calculate silhouette score
score = silhouette_score(
    X_scaled,
    model.predict(X),
    sample_size=5000,
    random_state=42
)

print("Silhouette Score:", round(score, 4))


Silhouette Score: 0.5853


In [7]:

import joblib

# Save trained model
joblib.dump(model, "energywatch_model.pkl")

# Save hourly data with predictions
hourly_df.to_csv("energywatch_data.csv", index=False)

print("Files saved successfully!")


Files saved successfully!


In [8]:

from google.colab import files

files.download("energywatch_model.pkl")
files.download("energywatch_data.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>